In [65]:
import os
import json
import csv
import textwrap
from pathlib import Path
from datetime import datetime

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
# langchain_text_splitters에서 직접 임포트
from langchain_text_splitters import RecursiveCharacterSplitter

from sklearn.metrics.pairwise import cosine_similarity

ImportError: cannot import name 'RecursiveCharacterSplitter' from 'langchain_text_splitters' (/usr/local/lib/python3.12/dist-packages/langchain_text_splitters/__init__.py)

In [66]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
# langchain_text_splitters에서 직접 임포트

In [20]:

from sklearn.metrics.pairwise import cosine_similarity

In [5]:
from google.colab import userdata
import os

# Colab Secrets에서 'OPENAI_API_KEY'를 가져옵니다.
try:
    api_key = userdata.get('OPENAI_API_KEY')
    os.environ['OPENAI_API_KEY'] = api_key
    print(f'Successfully loaded API key: {api_key[:10]}...')
except Exception as e:
    print('Colab Secrets에서 OPENAI_API_KEY를 찾을 수 없습니다. 왼쪽 열쇠 아이콘에서 설정을 확인해주세요.')

Successfully loaded API key: sk-proj-__...


In [6]:
!pip install -U langchain-openai langchain-text-splitters

In [4]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

In [8]:
llm = ChatOpenAI(model = 'gpt-4o-mini')

In [7]:
embedding = OpenAIEmbeddings(model = 'text-embedding-3-small')

In [9]:
SAMPLE_DIR = Path('sample_data')
SAMPLE_DIR.mkdir(exist_ok=True)

In [10]:
sample_texts = {
    'company_policy.txt': """주식회사 모두의연구소 사내 규정

제1조 (목적)
이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.

제2조 (근무시간)
기본 근무시간은 오전 9시부터 오후 6시까지로 한다.
유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.
재택근무는 주 2회까지 가능하다.

제3조 (휴가)
연차휴가는 근로기준법에 따라 부여한다.
경조사 휴가는 별도 규정에 따른다.
자기개발 휴가를 연 5일 추가 부여한다.

제4조 (교육)
모든 임직원은 연간 40시간 이상의 교육을 이수해야 한다.
외부 컨퍼런스 참석비를 연 200만원까지 지원한다.
온라인 학습 플랫폼 이용료를 전액 지원한다.
""",
    'ai_report.txt': """2024년 인공지능 산업 동향 보고서

개요
2024년 인공지능 산업은 전년 대비 35% 성장하여 약 5,000억 달러 규모에 도달했다.
특히 생성형 AI 분야가 전체 성장의 60%를 견인했다.

주요 트렌드
RAG(Retrieval-Augmented Generation): 기업용 AI 솔루션의 핵심 기술로 자리잡았다.
멀티모달 AI: 텍스트, 이미지, 음성을 통합 처리하는 모델이 확산되었다.
AI 에이전트: 자율적으로 작업을 수행하는 AI 에이전트 시장이 급성장했다.
소형 언어 모델(SLM): 경량화된 모델로 온디바이스 AI가 확대되었다.

시장 전망
2025년에는 AI 산업이 약 7,000억 달러 규모로 성장할 것으로 예상된다.
특히 RAG 기반 엔터프라이즈 솔루션 시장이 크게 확대될 전망이다.
""",
    'product_manual.txt': """스마트 홈 허브 v3.0 사용자 매뉴얼

제품 소개
스마트 홈 허브 v3.0은 AI 기반 홈 자동화 컨트롤러입니다.
음성 인식, 자동 스케줄링, 에너지 최적화 기능을 제공합니다.

초기 설정
Step 1: 전원을 연결하고 Wi-Fi 네트워크에 접속합니다.
Step 2: 모바일 앱을 설치하고 QR 코드를 스캔합니다.
Step 3: 연동할 IoT 기기를 검색하고 등록합니다.

주요 기능
음성 명령: "허브야, 거실 조명 켜줘" 등의 자연어 명령 지원
자동 스케줄: 시간대별 기기 자동 제어
에너지 모니터링: 실시간 전력 사용량 확인 및 절약 팁 제공
보안 모드: 외출 시 자동 보안 설정
"""
}

In [11]:
for filename, content in sample_texts.items():
  (SAMPLE_DIR / filename).write_text(content, encoding='utf-8')
  # sample_data/product_manual.txt 등의 파일이 생성됨

In [12]:
csv_data = [
    {'이름': '김철수', '부서':'개발팀', '직급':'선임'},
    {'이름': '이영희', '부서':'영업팀', '직급':'대리'},
    {'이름': '박지민', '부서':'영업팀', '직급':'주임'},
]

with open(SAMPLE_DIR / 'employees.csv', 'w', encoding='utf-8') as f:
  writer = csv.DictWriter(f, fieldnames=['이름','부서','직급'])
  writer.writeheader()
  writer.writerows(csv_data)

### RAG: Retrieval Augmented Generation
- 검색 증강 기술
- 코사인 유사도 등을 이용해 검색해와야하는데, 작동이 원할하지 않을 수 있음
- 시간의 흐름에 대한 사항은 반영 못함(날짜 관련..)
### Fine-tuning
- 추가 교육을 시키는 것

In [13]:
# 환각(할루시네이션): GPT 등 생성형모델은 단어들을 토큰단위로 생성함. BPE..등,
# 생성할 토큰 후보들이 있을 텐데
# 예시: 이름이 ______ 1) 학생, 2) 김철수, 3) 앤드류, 4) 예쁩니다....등등

In [14]:
knowledge_base = [
    {'id': 1, 'content': '파이썬은 1991년 귀도 반 로섬이 만든 프로그래밍 언어입니다. 간결한 문법과 풍부한 라이브러리가 특징입니다.', 'source': 'programming_guide.txt'},
    {'id': 2, 'content': 'RAG는 Retrieval-Augmented Generation의 약자로, LLM이 외부 데이터를 참조하여 답변을 생성하는 기술입니다.', 'source': 'ai_glossary.txt'},
    {'id': 3, 'content': '벡터 데이터베이스는 텍스트를 숫자 벡터로 변환하여 저장하고, 유사도 검색을 빠르게 수행하는 데이터베이스입니다.', 'source': 'database_manual.txt'},
]

In [17]:
query = 'RAG 기술이 무엇인가요?'
embeddings = OpenAIEmbeddings(model = 'text-embedding-3-small')

In [22]:
def retrieve_by_similarity(query, documents):
  contents = [doc['content'] for doc in documents]
  query_vec = embeddings.embed_query(query)
  doc_vecs = embeddings.embed_documents(contents)

  scores = cosine_similarity([query_vec], doc_vecs)[0]

  ranked = sorted(zip(documents, scores), key=lambda x: x[1], reverse = True)
  return [(doc, float(score)) for doc, score in ranked]

retrieve_by_similarity(query,knowledge_base)

[({'id': 2,
   'content': 'RAG는 Retrieval-Augmented Generation의 약자로, LLM이 외부 데이터를 참조하여 답변을 생성하는 기술입니다.',
   'source': 'ai_glossary.txt'},
  0.5588798938199673),
 ({'id': 1,
   'content': '파이썬은 1991년 귀도 반 로섬이 만든 프로그래밍 언어입니다. 간결한 문법과 풍부한 라이브러리가 특징입니다.',
   'source': 'programming_guide.txt'},
  0.2585202785690729),
 ({'id': 3,
   'content': '벡터 데이터베이스는 텍스트를 숫자 벡터로 변환하여 저장하고, 유사도 검색을 빠르게 수행하는 데이터베이스입니다.',
   'source': 'database_manual.txt'},
  0.17947009569873237)]

In [23]:
# 오늘 할 것은, RAG 이전 앞단에서 전처리 하는것

In [29]:
# 파이썬 문법 점검 => class

# class classname:
#   def __init__():

#   def

class Calculator:
  def __init__(self): # __init__은 클래스의 인스턴스가 만들어질 때 자동으로 해야하는 것들을 작성
    self.history = []

  def add(self, a, b):
    result = a+b
    self.history.append(f'{a} + {b} = {result}')

    return result

  def get_history(self):
    return self.history

In [32]:
calc = Calculator()

In [33]:
calc.history

[]

In [34]:
calc.add(1, 2)

3

In [35]:
calc.get_history()

['1 + 2 = 3']

In [49]:
# RAG용 클래스
class DocumentStore:
  def __init__(self):
    self.documents = []
    self.next_id = 1

  # 문서를 계속 집어넣음
  def add(self, content, source='unknown'):
    self.documents.append({
        'id': self.next_id,
        'content' : content,
        'source': source
    })
    self.next_id += 1

  def count(self):
    # 저장된 문서의 개수를 보여줌
    return len(self.documents)

  def search(self, keyword):
    return [doc for doc in self.documents if keyword in doc['content']]

   # 쿼리와 가장 유사도가 높은 문서를 self.documents에서 찾아서 반환
  def retrieve(self, query): # top_k 를 쓴다면, 유사도가 가장 높은 k개 반환
    if not self.documents:
        return []

    # 1. 쿼리 임베딩 생성
    query_emb = embeddings.embed_query(query)

    # 2. 문서들의 content만 추출하여 리스트 생성 및 임베딩 생성
    contents = [doc['content'] for doc in self.documents]
    doc_embs = embeddings.embed_documents(contents)

    # 3. 유사도 계산
    scores = cosine_similarity([query_emb], doc_embs)[0]

    # 4. 결과 정렬 및 반환
    sorted_docs = sorted(zip(self.documents, scores), key=lambda x: x[1], reverse=True)

    return [(doc, float(score)) for doc, score in sorted_docs] # sorted_docs[:top_k]

### 1. `sorted(zip(self.documents, scores), key=lambda x: x[1], reverse=True)`

이 문장은 여러 단계로 나누어 이해할 수 있습니다.

*   **`zip(self.documents, scores)`**:
    두 개의 리스트를 같은 인덱스끼리 묶어 튜플의 쌍으로 만듭니다.
    - 예: `[(doc1, 0.8), (doc2, 0.5), ...]`

*   **`key=lambda x: x[1]`**:
    정렬 기준을 정하는 함수입니다. `x`는 `zip`으로 묶인 각 항목(튜플)을 의미하며, `x[1]`은 두 번째 요소인 **유사도 점수(score)**를 뜻합니다. 즉, 점수를 기준으로 정렬하라는 명령입니다.

*   **`reverse=True`**:
    기본은 오름차순(작은 순서)이지만, 유사도가 **높은 순서**대로 보기 위해 내림차순으로 정렬합니다.

---

### 2. `[(doc, float(score)) for doc, score in sorted_docs]`

이것은 **리스트 컴프리헨션(List Comprehension)**이라는 문법으로, 리스트를 효율적으로 새로 만드는 방법입니다.

*   **`for doc, score in sorted_docs`**:
    정렬된 결과인 `(문서, 점수)` 튜플 꾸러미에서 하나씩 값을 꺼냅니다.

*   **`[(doc, float(score)) ...]`**:
    꺼낸 값들로 새로운 튜플 리스트를 만듭니다. 이때 `score`를 명시적으로 `float` 타입으로 변환하여 더 깔끔한 데이터를 반환하게 됩니다.

---

### 코드 예시로 이해하기

```python
# 예시 데이터
docs = ['문서A', '문서B']
scores = [0.1, 0.9]

# 1. zip으로 묶고 점수 기준 정렬
# x[1]인 0.9와 0.1을 비교해서 역순(reverse) 정렬
zipped = zip(docs, scores)
sorted_res = sorted(zipped, key=lambda x: x[1], reverse=True)
# 결과: [('문서B', 0.9), ('문서A', 0.1)]

# 2. 리스트 컴프리헨션으로 최종 형태 가공
final = [(d, float(s)) for d, s in sorted_res]
print(final)
# 출력: [('문서B', 0.9), ('문서A', 0.1)]
```

In [50]:
store = DocumentStore()
store.add('파이썬은 데이터 분석에 좋다', 'guide.txt')
store.add('RAG는 검색증강 생성이다.', 'glossary.txt')

In [51]:
store.count()


2

In [52]:
store.search('파이썬')

[{'id': 1, 'content': '파이썬은 데이터 분석에 좋다', 'source': 'guide.txt'}]

In [54]:
store.retrieve('파이썬')

[({'id': 1, 'content': '파이썬은 데이터 분석에 좋다', 'source': 'guide.txt'},
  0.5091253224776364),
 ({'id': 2, 'content': 'RAG는 검색증강 생성이다.', 'source': 'glossary.txt'},
  0.10721586623025722)]

In [59]:
sample_text = "문자열 예시입니다."
# 근데 단어 기반으로 어떻게 저장해?

class WordCounter:
  def __init__(self):
    self.texts = []


  def add_text(self, text):
      self.texts.append(text)

  def count_words(self):
      return sum(len(t.split()) for t in self.texts)

  def most_common(self, n):
    all_words = []
    for t in self.texts:
      all_words.extend(t.split())
    return Counter(all_words).most_common(n)



In [58]:
wc = WordCounter()
wc.add_text('파이썬은 어렵지 않습니다.')

wc.add_text('나는 학교에 갑니다.')

wc.count_words()

6

In [60]:
# 많이 쓰는 라이브러리
from collections import Counter # Counter라는 클래스, 글자를 세는데 편하게 사용 가능

all_words = [1,1,1,1,1,2,2,2,2,3,3,3,4,4,5]
Counter(all_words)

Counter({1: 5, 2: 4, 3: 3, 4: 2, 5: 1})

In [ ]:
# 기존 DocumentStore은 유사 RAG긴 한데, 생성 기능이 없음

In [72]:
query = 'RAG에 대해 설명해주세요.'
docs = ['RAG는 검색증강생성 기술입니다.', '외부 문서를 검색해서 llm 답변에 활용합니다.']

# 쿼리를 입력하면 생성하지만, knowledge_base에서 가져온 문서(document)들을 LLM의 context로 넣음
def rag_with_langchain(query, documents): # document에는 참조할 문서
  if not documents:
    return "참고할 문서가 없습니다."
  contexts = '\n'.join(f"- {doc}" for doc in documents) # join은 리스트에 있는걸 연결
  messages = [
      SystemMessage(content = '제공된 문서를 기반으로 정확하게 답변해주세요.'),
      HumanMessage(content = f'참고 문서:\n{contexts}\n\n질문: {query}')
  ]
  response = llm.invoke(messages)
  return response.content # llm이 생성한 값

In [64]:
docs = ['RAG는 검색증강생성 기술입니다.', '외부 문서를 검색해서 llm 답변에 활용합니다.']
contexts = '\n'.join(f"- {doc}" for doc in docs)
contexts

'- RAG는 검색증강생성 기술입니다.\n- 외부 문서를 검색해서 llm 답변에 활용합니다.'

In [69]:
rag_with_langchain('rag가 뭐야?', docs)

'RAG는 "Retrieval-Augmented Generation"의 약자로, 검색증강생성 기술입니다. 이 기술은 외부 문서나 정보를 검색하여 이를 활용해 대규모 언어 모델(LLM)이 더 정확하고 유용한 답변을 생성하도록 돕습니다. 즉, 기본적으로 질문에 답하기 위해 외부 데이터 소스를 검색하고, 이 정보를 바탕으로 생성된 답변을 제공합니다.'

In [73]:
rag_with_langchain('rag가 뭐야?', [])

'참고할 문서가 없습니다.'

In [90]:
# RAG용 클래스
class DocumentStore:
  def __init__(self):
    self.documents = []
    self.next_id = 1

  # 문서를 계속 집어넣음
  def add(self, content, source='unknown'):
    self.documents.append({
        'id': self.next_id,
        'content' : content,
        'source': source
    })
    self.next_id += 1

  def count(self):
    # 저장된 문서의 개수를 보여줌
    return len(self.documents)

  def search(self, keyword):
    return [doc for doc in self.documents if keyword in doc['content']]

   # 쿼리와 가장 유사도가 높은 문서를 self.documents에서 찾아서 반환
  def retrieve(self, query, top_k): # top_k 를 쓴다면, 유사도가 가장 높은 k개 반환
    if not self.documents:
        return []

    # 1. 쿼리 임베딩 생성
    query_emb = embeddings.embed_query(query)

    # 2. 문서들의 content만 추출하여 리스트 생성 및 임베딩 생성
    contents = [doc['content'] for doc in self.documents]
    doc_embs = embeddings.embed_documents(contents)

    # 3. 유사도 계산
    scores = cosine_similarity([query_emb], doc_embs)[0]

    # 4. 결과 정렬 및 반환
    sorted_docs = sorted(zip(self.documents, scores), key=lambda x: x[1], reverse=True)

    return [(doc, float(score)) for doc, score in sorted_docs[:top_k]] # sorted_docs[:top_k]

  def generate(self, query):
    # document랑 query를 받아서 증강 생성을 해라
    similar_contents = []
    # retrieve된 결과를 사용해야함
    retrieved_docs = self.retrieve(query, top_k=3)
    similar_contents.append([content['content'] for (content, score) in retrieved_docs])

    contexts = '\n'.join(f"- {doc}" for doc in similar_contents) # join은 리스트에 있는걸 연결

    messages = [
        SystemMessage(content = '제공된 문서를 기반으로 정확하게 답변해주세요.'),
        HumanMessage(content = f'참고 문서:\n{contexts}\n\n질문: {query}')
    ]
    response = llm.invoke(messages)
    return response

In [93]:
ds = DocumentStore()

In [94]:
ds.generate('안녕하세요, 치킨은 누가 처음 만들었나요?')

AIMessage(content='안녕하세요! 치킨은 특별히 한 사람이 처음 만든 음식은 아닙니다. 치킨 요리는 여러 문화에서 오랜 역사를 가지고 있으며, 인류가 가축을 기르기 시작한 이후로 오래전부터 다양한 방식으로 조리되고 소비되었습니다. 특히 아시아, 유럽, 아메리카 등지에서 각각 독특한 방식으로 발전해왔습니다. 따라서 치킨 요리의 기원은 다양한 요인에 의해 형성된 복합적인 역사라고 할 수 있습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 108, 'prompt_tokens': 47, 'total_tokens': 155, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ca3e7d71bf', 'id': 'chatcmpl-DL74kFwz2Z6VQrlyc9ETYTqFfvQ0K', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d0623-20cc-7e43-89c8-2d01a0a3408f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 47, 'output_tokens': 108, 'total_tokens': 155, 'input_token_details': {'au

In [88]:
ds.add('치킨은 처음 켄터키 할아버지가 만드셨다고 들었습니다. 1900년대 중반 켄터키 지방에서 압력솥으로 만들었다고 합니다.')

In [92]:
ds.generate('안녕하세요, 치킨은 누가 처음 만들었나요?')

[SystemMessage(content='제공된 문서를 기반으로 정확하게 답변해주세요.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content="참고 문서:\n- ['치킨은 처음 켄터키 할아버지가 만드셨다고 들었습니다. 1900년대 중반 켄터키 지방에서 압력솥으로 만들었다고 합니다.']\n\n질문: 안녕하세요, 치킨은 누가 처음 만들었나요?", additional_kwargs={}, response_metadata={})]

강사님 코드

In [101]:
# RAG용 클래스
class DocumentStore:
  def __init__(self):
    self.documents = []
    self.next_id = 1

  # 문서를 계속 집어넣음
  def add(self, content, source='unknown'):
    self.documents.append({
        'id': self.next_id,
        'content' : content,
        'source': source
    })
    self.next_id += 1

  def count(self):
    # 저장된 문서의 개수를 보여줌
    return len(self.documents)

  def search(self, keyword):
    return [doc for doc in self.documents if keyword in doc['content']]

   # 쿼리와 가장 유사도가 높은 문서를 self.documents에서 찾아서 반환
  def retrieve(self, query, top_k): # top_k 를 쓴다면, 유사도가 가장 높은 k개 반환
    if not self.documents:
        return []

    # 1. 쿼리 임베딩 생성
    query_emb = embeddings.embed_query(query)

    # 2. 문서들의 content만 추출하여 리스트 생성 및 임베딩 생성
    contents = [doc['content'] for doc in self.documents]
    doc_embs = embeddings.embed_documents(contents)

    # 3. 유사도 계산
    scores = cosine_similarity([query_emb], doc_embs)[0]

    # 4. 결과 정렬 및 반환
    sorted_docs = sorted(zip(self.documents, scores), key=lambda x: x[1], reverse=True)

    return [(doc, float(score)) for doc, score in sorted_docs[:top_k]] # sorted_docs[:top_k]

  def generate(self, query):
    retrieved = self.retrieve(query, 3)
    if not retrieved:
      return '관련 문서를 찾을 수 없습니다.'

    contexts = '\n'.join(f"[문서 {d[0]['id']}, 출처:{d[0]['source']}, {d[0]['content']}]" for d in retrieved) # join은 리스트에 있는걸 연결

    messages = [
        SystemMessage(content = '제공된 문서를 기반으로 정확하게 답변해주세요. 출처를 명시해주세요'),
        HumanMessage(content = f'참고 문서:\n{contexts}\n\n질문: {query}')
    ]
    response = llm.invoke(messages)
    return response.content


In [102]:
store = DocumentStore()
store.add('파이썬은 데이터 분석에 좋다', 'guide.txt')
store.add('RAG는 검색증강 생성이다.', 'glossary.txt')

In [103]:
store.generate('rag에 대해 설명해주세요.')

'RAG는 "검색증강 생성"을 의미합니다. 이는 정보 검색과 생성 모델을 결합하여 보다 효율적으로 정보를 생성하는 방법론입니다. 이 방법은 사용자가 제공하는 질문이나 요청에 대한 정확하고 관련성 높은 정보를 검색한 후, 그 정보를 바탕으로 새롭고 유용한 내용을 생성하는 방식입니다. 출처: [문서 2, glossary.txt]'

In [104]:
# 문서로딩

from langchain_core.documents import Document

In [105]:
doc = Document(page_content = "문서 내용...", metadata = {"source":"file.txt", "type":"policy"})
file_path = SAMPLE_DIR / 'company_policy.txt'

with open(file_path, 'r', encoding = 'utf-8') as f:
  raw_text = f.read()

In [106]:
raw_text

'주식회사 모두의연구소 사내 규정\n\n제1조 (목적)\n이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.\n\n제2조 (근무시간)\n기본 근무시간은 오전 9시부터 오후 6시까지로 한다.\n유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.\n재택근무는 주 2회까지 가능하다.\n\n제3조 (휴가)\n연차휴가는 근로기준법에 따라 부여한다.\n경조사 휴가는 별도 규정에 따른다.\n자기개발 휴가를 연 5일 추가 부여한다.\n\n제4조 (교육)\n모든 임직원은 연간 40시간 이상의 교육을 이수해야 한다.\n외부 컨퍼런스 참석비를 연 200만원까지 지원한다.\n온라인 학습 플랫폼 이용료를 전액 지원한다.\n'

In [110]:
file_path.name

'company_policy.txt'

In [112]:
doc = Document(page_content = raw_text, metadata = {"source":file_path.name,
                                                    "type":"txt",
                                                    "char_count": len(raw_text),
                                                    "line_count": len(raw_text.splitlines())



                                                    })

In [115]:
doc

Document(metadata={'source': 'company_policy.txt', 'type': 'txt', 'char_count': 356, 'line_count': 19}, page_content='주식회사 모두의연구소 사내 규정\n\n제1조 (목적)\n이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.\n\n제2조 (근무시간)\n기본 근무시간은 오전 9시부터 오후 6시까지로 한다.\n유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.\n재택근무는 주 2회까지 가능하다.\n\n제3조 (휴가)\n연차휴가는 근로기준법에 따라 부여한다.\n경조사 휴가는 별도 규정에 따른다.\n자기개발 휴가를 연 5일 추가 부여한다.\n\n제4조 (교육)\n모든 임직원은 연간 40시간 이상의 교육을 이수해야 한다.\n외부 컨퍼런스 참석비를 연 200만원까지 지원한다.\n온라인 학습 플랫폼 이용료를 전액 지원한다.\n')

In [116]:
doc.page_content

'주식회사 모두의연구소 사내 규정\n\n제1조 (목적)\n이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.\n\n제2조 (근무시간)\n기본 근무시간은 오전 9시부터 오후 6시까지로 한다.\n유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.\n재택근무는 주 2회까지 가능하다.\n\n제3조 (휴가)\n연차휴가는 근로기준법에 따라 부여한다.\n경조사 휴가는 별도 규정에 따른다.\n자기개발 휴가를 연 5일 추가 부여한다.\n\n제4조 (교육)\n모든 임직원은 연간 40시간 이상의 교육을 이수해야 한다.\n외부 컨퍼런스 참석비를 연 200만원까지 지원한다.\n온라인 학습 플랫폼 이용료를 전액 지원한다.\n'

In [131]:
def load_text_files(directory):
  documents = []
  for fp in sorted(directory.glob('*.txt')):
    text = fp.read_text(encoding='utf-8')
    doc = Document(page_content = text,
                   metadata = {'source': fp.name,
                               'char_count' : len(text),
                               }
                   )
    documents.append(doc)

  return documents

In [132]:
for fp in SAMPLE_DIR.glob('*.txt'):
  print(fp) # glob(): txt로 끝나는 모든 파일들을 리스트로 가져오는 것임

sample_data/product_manual.txt
sample_data/ai_report.txt
sample_data/company_policy.txt


In [133]:
all_docs = load_text_files(SAMPLE_DIR)

In [134]:
all_docs

[Document(metadata={'source': 'ai_report.txt', 'char_count': 396}, page_content='2024년 인공지능 산업 동향 보고서\n\n개요\n2024년 인공지능 산업은 전년 대비 35% 성장하여 약 5,000억 달러 규모에 도달했다.\n특히 생성형 AI 분야가 전체 성장의 60%를 견인했다.\n\n주요 트렌드\nRAG(Retrieval-Augmented Generation): 기업용 AI 솔루션의 핵심 기술로 자리잡았다.\n멀티모달 AI: 텍스트, 이미지, 음성을 통합 처리하는 모델이 확산되었다.\nAI 에이전트: 자율적으로 작업을 수행하는 AI 에이전트 시장이 급성장했다.\n소형 언어 모델(SLM): 경량화된 모델로 온디바이스 AI가 확대되었다.\n\n시장 전망\n2025년에는 AI 산업이 약 7,000억 달러 규모로 성장할 것으로 예상된다.\n특히 RAG 기반 엔터프라이즈 솔루션 시장이 크게 확대될 전망이다.\n'),
 Document(metadata={'source': 'company_policy.txt', 'char_count': 356}, page_content='주식회사 모두의연구소 사내 규정\n\n제1조 (목적)\n이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.\n\n제2조 (근무시간)\n기본 근무시간은 오전 9시부터 오후 6시까지로 한다.\n유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.\n재택근무는 주 2회까지 가능하다.\n\n제3조 (휴가)\n연차휴가는 근로기준법에 따라 부여한다.\n경조사 휴가는 별도 규정에 따른다.\n자기개발 휴가를 연 5일 추가 부여한다.\n\n제4조 (교육)\n모든 임직원은 연간 40시간 이상의 교육을 이수해야 한다.\n외부 컨퍼런스 참석비를 연 200만원까지 지원한다.\n온라인 학습 플랫폼 이용료를 전액 지원한다.\n'),
 Document(metadata={'source': 'product_

In [135]:
def load_csv_file(file_path):
  documents = []
  with open(file_path, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for i, row in enumerate(reader):
      content = ' | '.join(f'{k}:{v}' for k, v in row.items())
      doc = Document(page_content = content,
                    metadata = {'source': file_path.name,
                                'char_count' : len(content),
                                }
                    )
      documents.append(doc)

    return documents

In [136]:
csv_docs = load_csv_file(SAMPLE_DIR / 'employees.csv')
csv_docs

[Document(metadata={'source': 'employees.csv', 'char_count': 23}, page_content='이름:김철수 | 부서:개발팀 | 직급:선임'),
 Document(metadata={'source': 'employees.csv', 'char_count': 23}, page_content='이름:이영희 | 부서:영업팀 | 직급:대리'),
 Document(metadata={'source': 'employees.csv', 'char_count': 23}, page_content='이름:박지민 | 부서:영업팀 | 직급:주임')]

### JSON

In [137]:
import json

In [141]:
test_json = {
    'name': 'RAG 프로젝트',
    'version': '1.0',
    'features': ['검색', '생성']

}

with open('sample_data/test.json', 'w', encoding='utf-8') as f:
  json.dump(test_json, f, ensure_ascii=False)

In [143]:
with open('sample_data/test.json', 'r', encoding='utf-8') as f:
  data = json.load(f)

# dumps(string으로 반환)를 사용해야 합니다.
content = json.dumps(data, ensure_ascii = False, indent = 2)
print(content)

{
  "name": "RAG 프로젝트",
  "version": "1.0",
  "features": [
    "검색",
    "생성"
  ]
}


In [144]:
data

{'name': 'RAG 프로젝트', 'version': '1.0', 'features': ['검색', '생성']}

In [145]:
content

'{\n  "name": "RAG 프로젝트",\n  "version": "1.0",\n  "features": [\n    "검색",\n    "생성"\n  ]\n}'

In [146]:
doc = Document(page_content = content, metadata = {'source': 'test.json', 'keys': data.keys()})

In [147]:
doc

Document(metadata={'source': 'test.json', 'keys': dict_keys(['name', 'version', 'features'])}, page_content='{\n  "name": "RAG 프로젝트",\n  "version": "1.0",\n  "features": [\n    "검색",\n    "생성"\n  ]\n}')

In [148]:
def load_json_as_document(file_path):
  path = Path(file_path)
  with open(path, 'r', encoding='utf-8') as f:
    data = json.load(f)

  content = json.dumps(data, ensure_ascii=False, indent=2)
  keys = list(data.keys())

  return Document(page_content = content, metadata = {'source':path.name, 'keys': keys})

## 📝 오늘 학습 내용 요약

### 1. RAG (Retrieval-Augmented Generation) 핵심 개념
- **할루시네이션(환각)**: LLM이 사실이 아닌 내용을 그럴듯하게 답변하는 현상을 방지하기 위해 외부 지식을 참조합니다.
- **검색(Retrieval)**: 질문과 유사도가 높은 문서를 임베딩과 코사인 유사도를 이용해 찾아냅니다.
- **생성(Generation)**: 검색된 문서를 컨텍스트로 활용하여 답변을 생성합니다.

### 2. Python 클래스를 활용한 시스템 구축
- **DocumentStore**: 문서를 저장(`add`), 키워드 검색(`search`), 유사도 기반 검색(`retrieve`), 그리고 최종 답변 생성(`generate`) 기능을 포함하는 클래스를 직접 설계해 보았습니다.
- **인스턴스화**: `store = DocumentStore()`와 같이 클래스를 통해 객체를 생성하고 내부 변수(`self`)를 관리하는 법을 익혔습니다.

### 3. 데이터 전처리 및 로딩 (Data Loading)
- **Document 객체**: LangChain에서 사용하는 표준 문서 규격인 `page_content`(텍스트)와 `metadata`(부가 정보)의 구조를 이해했습니다.
- **파일 형식별 처리**:
    - **TXT**: `Path` 객체와 `glob`를 이용한 일괄 로딩
    - **CSV**: `DictReader`를 활용해 행(Row) 데이터를 문자열로 변환하여 로딩
    - **JSON**: `json.load()`로 읽은 뒤 `json.dumps()`를 통해 문자열화하여 저장하는 차이점(객체 vs 문자열)을 학습했습니다.

### 4. 주요 문법 포인트
- **zip & lambda**: 두 리스트를 묶고 특정 인덱스(유사도 점수)를 기준으로 정렬하는 기법
- **List Comprehension**: 튜플 언패킹(`for doc, score in ...`)을 활용해 리스트를 효율적으로 가공하는 방법